# DIR LOADER

This is a directory loader which use RAG retriver to compress and load code in directory for agent to have a through check for further sub agent process.

In [6]:
# see lab 4 for detail

import os
import json
import logging
from pathlib import Path
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Setup logging
logging.basicConfig(
    format='[%(asctime)s] p%(process)s {%(filename)s:%(lineno)d} %(levelname)s - %(message)s',
    level=logging.INFO
)

logger = logging.getLogger(__name__)
if os.getenv("GROQ_API_KEY"):
    logger.info("GROQ_API_KEY is set")

[2026-03-16 17:39:17,131] p14608 {3440689807.py:20} INFO - GROQ_API_KEY is set


In [7]:
from langchain_community.chat_models import ChatLiteLLM

# Configure the LLM - using Groq's free Llama model
# You can change this to other models like:
# - "groq/llama-3.3-70b-versatile" (larger, FREE)
# - "gpt-4o-mini" (OpenAI, paid)
# - "claude-3-5-haiku-20241022" (Anthropic, paid)

MODEL_ID = "groq/llama-3.1-8b-instant"

llm = ChatLiteLLM(
    model=MODEL_ID,
    temperature=0
)
logger.info(f"Using model: {MODEL_ID}")

[2026-03-16 17:39:17,144] p14608 {4098160048.py:15} INFO - Using model: groq/llama-3.1-8b-instant


In [8]:
response = llm.invoke(input="What is this directory about?")
logger.info(response.content)

17:39:17 - LiteLLM:INFO: utils.py:2699 - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-03-16 17:39:17,156] p14608 {utils.py:2699} INFO - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-03-16 17:39:17,415] p14608 {_client.py:1038} INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
17:39:17 - LiteLLM:INFO: utils.py:909 - Wrapper: Completed Call, calling success_handler
[2026-03-16 17:39:17,418] p14608 {utils.py:909} INFO - Wrapper: Completed Call, calling success_handler
[2026-03-16 17:39:17,423] p14608 {2723229751.py:2} INFO - This conversation has just started. I'm happy to chat with you, but I don't see any directory information. Could you please provide more context or information about the directory you're referring to? I'll do my best to help.


In here we use a github project which contain old python code for test, the goal is to read and understand the project and prepear for further sub agent analysis

In [9]:
!ls -ltr old-demos
!ls -ltr old-demos/demo

total 48
-rw-r--r--   1 hongchongyuan  staff  13930 Mar 16 17:09 LICENSE
-rw-r--r--   1 hongchongyuan  staff   5944 Mar 16 17:09 README.md
drwxr-xr-x  16 hongchongyuan  staff    512 Mar 16 17:09 demo
drwxr-xr-x  82 hongchongyuan  staff   2624 Mar 16 17:09 scripts
total 224
-rw-r--r--  1 hongchongyuan  staff   1049 Mar 16 17:09 README
-rwxr-xr-x  1 hongchongyuan  staff    566 Mar 16 17:09 beer.py
-rwxr-xr-x  1 hongchongyuan  staff   3911 Mar 16 17:09 eiffel.py
-rwxr-xr-x  1 hongchongyuan  staff   4611 Mar 16 17:09 hanoi.py
-rwxr-xr-x  1 hongchongyuan  staff   8987 Mar 16 17:09 life.py
-rwxr-xr-x  1 hongchongyuan  staff   3691 Mar 16 17:09 markov.py
-rwxr-xr-x  1 hongchongyuan  staff   2223 Mar 16 17:09 mcast.py
-rwxr-xr-x  1 hongchongyuan  staff   2270 Mar 16 17:09 queens.py
-rwxr-xr-x  1 hongchongyuan  staff   5749 Mar 16 17:09 redemo.py
-rwxr-xr-x  1 hongchongyuan  staff    811 Mar 16 17:09 rpython.py
-rwxr-xr-x  1 hongchongyuan  staff   1323 Mar 16 17:09 rpythond.py
-rwxr-xr-x  1 hon

## Langchain directory loader (from lab 4)

In [10]:
import importlib.util
print("markdown:", importlib.util.find_spec("markdown"))
print("yaml:", importlib.util.find_spec("yaml"))
print("bs4:", importlib.util.find_spec("bs4"))

markdown: ModuleSpec(name='markdown', loader=<_frozen_importlib_external.SourceFileLoader object at 0x1006eaf30>, origin='/Users/hongchongyuan/Desktop/DSAN_spring_2026/dsan6750/spring-2026-final-project-team_026/.venv/lib/python3.12/site-packages/markdown/__init__.py', submodule_search_locations=['/Users/hongchongyuan/Desktop/DSAN_spring_2026/dsan6750/spring-2026-final-project-team_026/.venv/lib/python3.12/site-packages/markdown'])
yaml: ModuleSpec(name='yaml', loader=<_frozen_importlib_external.SourceFileLoader object at 0x122ad2450>, origin='/Users/hongchongyuan/Desktop/DSAN_spring_2026/dsan6750/spring-2026-final-project-team_026/.venv/lib/python3.12/site-packages/yaml/__init__.py', submodule_search_locations=['/Users/hongchongyuan/Desktop/DSAN_spring_2026/dsan6750/spring-2026-final-project-team_026/.venv/lib/python3.12/site-packages/yaml'])
bs4: ModuleSpec(name='bs4', loader=<_frozen_importlib_external.SourceFileLoader object at 0x143a3dfd0>, origin='/Users/hongchongyuan/Desktop/DSA

In [ ]:
from langchain_community.document_loaders import DirectoryLoader
# This use "unstructured[md]" which allow .py file reading

loader = DirectoryLoader("old-demos/", glob=["**/*.txt", "**/*.md", "**/*.py", "**/*.sh"]) # read EVERYTHING!
documents = loader.load()
logger.info(f"there are {len(documents)} files in this directory")

[2026-03-16 17:39:25,456] p14608 {filetype.py:435} WARNING - libmagic is unavailable but assists in filetype detection. Please consider installing libmagic for better results.
[2026-03-16 17:39:49,650] p14608 {filetype.py:435} WARNING - libmagic is unavailable but assists in filetype detection. Please consider installing libmagic for better results.
[2026-03-16 17:39:49,660] p14608 {filetype.py:435} WARNING - libmagic is unavailable but assists in filetype detection. Please consider installing libmagic for better results.
[2026-03-16 17:39:49,924] p14608 {filetype.py:435} WARNING - libmagic is unavailable but assists in filetype detection. Please consider installing libmagic for better results.
[2026-03-16 17:39:49,940] p14608 {filetype.py:435} WARNING - libmagic is unavailable but assists in filetype detection. Please consider installing libmagic for better results.
[2026-03-16 17:39:49,968] p14608 {filetype.py:435} WARNING - libmagic is unavailable but assists in filetype detection. 

In [12]:
logger.info(documents[0])

[2026-03-16 17:40:49,851] p14608 {3836738686.py:1} INFO - page_content='Very old Python demos and examples

These files were rescued from the cpython repo, where they used to live in Tools/demo/ and Tools/scripts/.

Demo

This directory contains a collection of demonstration scripts for various aspects of Python programming.

beer.py           Well-known programming example: Bottles of beer.
eiffel.py         Python advanced magic: A metaclass for Eiffel post/preconditions.
hanoi.py          Well-known programming example: Towers of Hanoi.
life.py           Curses programming: Simple game-of-life.
markov.py         Algorithms: Markov chain simulation.
mcast.py          Network programming: Send and receive UDP multicast packets.
queens.py         Well-known programming example: N-Queens problem.
redemo.py         Regular Expressions: GUI script to test regexes.
rpython.py        Network programming: Small client for remote code execution.
rpythond.py       Network programming: Small se

## Chunking retriver

In [13]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(documents)

In [14]:
logger.info(f"we converted {len(documents)} documents into {len(splits)} chunks")

[2026-03-16 17:41:32,340] p14608 {209985795.py:1} INFO - we converted 88 documents into 582 chunks


In [15]:
splits[0]

Document(metadata={'source': 'old-demos/README.md'}, page_content='Very old Python demos and examples\n\nThese files were rescued from the cpython repo, where they used to live in Tools/demo/ and Tools/scripts/.\n\nDemo\n\nThis directory contains a collection of demonstration scripts for various aspects of Python programming.')

## Indexing

In [17]:
import faiss
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

# Using a free, local embedding model
embeddings_model = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)
logger.info("Embeddings model loaded successfully")

/var/folders/0q/22f144kx4dn41cw151l589xh0000gn/T/ipykernel_14608/2957083136.py:6: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings_model = HuggingFaceEmbeddings(
/Users/hongchongyuan/Desktop/DSAN_spring_2026/dsan6750/spring-2026-final-project-team_026/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[2026-03-16 17:46:10,557] p14608 {SentenceTransformer.py:219} INFO - Use pytorch device_name: mps
[2026-03-16 17:46:10,558] p14608 {SentenceTransformer.py:227} I

In [18]:
vectorstore = FAISS.from_documents(documents=splits, embedding=embeddings_model)

Now directory with code init is already been compress and chunked for agent to retrive

In [28]:
retriever = vectorstore.as_retriever(search_kwargs={'k': 10})
retrieved_docs = retriever.invoke(
    "What is this git hub directory about?"
)
logger.info(f"Number of retrieved documents: {len(retrieved_docs)}")
logger.info(retrieved_docs[0].page_content)

[2026-03-16 18:01:38,990] p14608 {461611952.py:5} INFO - Number of retrieved documents: 10
[2026-03-16 18:01:38,999] p14608 {461611952.py:6} INFO - Note that the history of these files was preserved from the cpython repo. The LICENSE file is copied from that repo, it applies to these files.

Scripts

This directory contains a collection of executable Python scripts that are useful while building, extending or managing Python. Some (e.g., dutree or lll) are also generally useful UNIX tools.


In [29]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

system_prompt = (
    "You are an agent aim to solve tech debt problem in directory you loaded, that the directory you read have many outdated code"
    "Use the following pieces of retrieved context to understand the main picture of code "
    "If you don't know the answer, say that you "
    "don't know! "
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)
results = rag_chain.invoke(
    {"input": "Summarize the directory, what is it about?"}
)
logger.info(results["answer"])

18:01:56 - LiteLLM:INFO: utils.py:2699 - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-03-16 18:01:56,224] p14608 {utils.py:2699} INFO - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-03-16 18:01:57,199] p14608 {_client.py:1038} INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
18:01:57 - LiteLLM:INFO: utils.py:909 - Wrapper: Completed Call, calling success_handler
[2026-03-16 18:01:57,207] p14608 {utils.py:909} INFO - Wrapper: Completed Call, calling success_handler
[2026-03-16 18:01:57,226] p14608 {4260334112.py:25} INFO - The directory appears to be a collection of Python scripts and tools related to Python development, maintenance, and management. The scripts are designed to perform various tasks such as:

1. File statistics and analysis
2. HTML Help generation
3. Symbolic link management
4. Code processing and modification
5. Error message parsing
6. MD5 checksum calculation
7. Intellige

In [30]:
retriever = vectorstore.as_retriever(search_kwargs={'k': 3})
rag_chain = create_retrieval_chain(retriever, question_answer_chain)
results = rag_chain.invoke(
    {"input": "In beer.py which part could be outdated or using old packages which no longer availiable in python 3.10? "}
)
logger.info(results["answer"])

18:02:29 - LiteLLM:INFO: utils.py:2699 - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-03-16 18:02:29,316] p14608 {utils.py:2699} INFO - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-03-16 18:02:30,269] p14608 {_client.py:1038} INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
18:02:30 - LiteLLM:INFO: utils.py:909 - Wrapper: Completed Call, calling success_handler
[2026-03-16 18:02:30,279] p14608 {utils.py:909} INFO - Wrapper: Completed Call, calling success_handler
[2026-03-16 18:02:30,288] p14608 {721137519.py:6} INFO - After reviewing the code in `beer.py`, I don't see any obvious outdated or deprecated code that would prevent it from running in Python 3.10. However, here are a few potential areas that could be considered outdated or using old packages:

1. **Python 2 compatibility**: Although the shebang line `#!/usr/bin/env python3` suggests that the code is intended for Python 3, the c

## retriver tech

In [32]:
retriever = vectorstore.as_retriever(search_type = "mmr", search_kwargs={'k': 10})
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

results = rag_chain.invoke(
            {"input": "Summarize the directory, what is it about?"}
        )
logger.info(results["answer"])

18:40:18 - LiteLLM:INFO: utils.py:2699 - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-03-16 18:40:18,686] p14608 {utils.py:2699} INFO - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
[2026-03-16 18:40:19,786] p14608 {_client.py:1038} INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
18:40:19 - LiteLLM:INFO: utils.py:909 - Wrapper: Completed Call, calling success_handler
[2026-03-16 18:40:19,790] p14608 {utils.py:909} INFO - Wrapper: Completed Call, calling success_handler
[2026-03-16 18:40:19,796] p14608 {1377251674.py:8} INFO - The directory appears to be a collection of Python scripts and tools that were likely used for building, extending, or managing the Python programming language. The scripts are a mix of utility tools, documentation generators, and testing tools.

Some of the notable scripts and tools in the directory include:

1. `finddiv`: a grep-like tool for finding division operators i